# AI 드론 데이터 코딩 실습 노트 (실습용)
### Ⅳ. AI 활용 프로그래밍

이 노트는 교재의 예제를 **직접 채워 넣으며** 익히는 실습용입니다.

1. 셀을 **위에서부터 차례대로** 실행합니다. (**Shift + Enter**)
2. 코드 셀에서 `#` 설명을 읽고 **비어 있는 곳**(`=` 뒤, 설명 아래 빈 줄, 빈 괄호)을 채웁니다.
3. 셀 아래 **실행 결과**와 같게 나오면 성공입니다. 막히면 교재의 같은 번호 예제를 봅니다.
4. `해 보기`는 정답이 하나가 아닙니다. 바꿔 보고, 결과가 왜 달라졌는지 말해 봅시다.

> 빈칸을 채우기 전에 실행하면 빨간 오류가 나옵니다. 고장이 아니라 '아직 덜 썼다'는 뜻입니다.

| 소단원 | 예제 |
|---|---|
| 4-1 AI 활용 원칙 | (읽기·토의) |
| 4-2 프롬프트 작성과 코드 생성 | 4-1 |
| 4-3 AI 생성 결과 검증 및 개선 | 4-2, 4-3, 4-4 |
| 4-4 기계학습 기초와 결정트리 활용 | 4-5, 4-6, 4-7, 4-8, 4-9 |

## 0. 준비하기

아래 셀은 **실습 준비용**입니다. 내용을 이해하지 않아도 괜찮습니다. 실행만 하면 다음 일을 대신 해 줍니다.

- 가상 드론 `drone_sim.py`를 인터넷에서 내려받습니다. 내려받지 못했을 때만 **파일 선택 창**이 뜹니다. 그때는 선생님이 나누어 준 파일을 골라 올립니다.
- 그래프에 한글이 깨지지 않게 해 주는 `font_setup.py`를 만들어 둡니다.

> 코랩은 연결이 끊기면 파일을 잊습니다. 다음 시간에 다시 열었다면 이 셀부터 다시 실행합니다.

In [ ]:
#@title ▶ 준비 셀 (실행만 하세요)
import os, importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# 1) 한글 글꼴 설정 모듈(font_setup.py)을 만들어 둔다 (교재 부록 2와 같은 내용)
FONT_SETUP = '''"""font_setup.py - 그래프에 한글이 깨지지 않도록 글꼴을 정해 주는 모듈
그래프를 그리는 예제에서 import font_setup 한 줄만 적으면 된다.
코랩에는 한글 글꼴이 없으므로, 처음 한 번 나눔고딕을 설치한다.
"""
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NANUM = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# 1) 코랩이면 나눔고딕을 설치해서 등록한다
if os.path.exists("/content") and not os.path.exists(NANUM):
    os.system("apt-get -qq install fonts-nanum > /dev/null")
if os.path.exists(NANUM):
    fm.fontManager.addfont(NANUM)

# 2) 운영체제마다 들어 있는 한글 글꼴이 다르므로, 있는 것을 찾아 쓴다
candidates = ["NanumGothic", "Malgun Gothic", "AppleGothic", "Noto Sans CJK KR"]
installed = {f.name for f in fm.fontManager.ttflist}
for name in candidates:
    if name in installed:
        plt.rcParams["font.family"] = name
        break
else:
    print("한글 글꼴을 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")

plt.rcParams["axes.unicode_minus"] = False   # 음수 기호가 깨지지 않도록
'''
if not os.path.exists("font_setup.py"):
    with open("font_setup.py", "w", encoding="utf-8") as f:
        f.write(FONT_SETUP)

# 2) 가상 드론(drone_sim.py)을 내려받는다. 받지 못하면 파일 선택 창을 띄운다
import urllib.request
URL = "https://raw.githubusercontent.com/ralralra/dronebook/main/data/drone_sim.py"
try:
    urllib.request.urlretrieve(URL, "drone_sim.py")
except Exception:
    print("drone_sim.py 를 내려받지 못했습니다.")
if not os.path.exists("drone_sim.py") and IN_COLAB:
    from google.colab import files
    print("선생님이 나누어 준 drone_sim.py 를 골라 올려 주세요.")
    files.upload()

for name in ["drone_sim.py", "font_setup.py"]:
    print(("✓ 준비됨  " if os.path.exists(name) else "✗ 없음    ") + name)


# 4-1 AI 활용 원칙

**학습 목표**

- 생성형 AI가 답을 만들어 내는 방식과 그 한계를 설명할 수 있습니다.
- AI를 활용할 때 지켜야 할 원칙을 설명하고 실천할 수 있습니다.
- AI를 활용한 결과에 대한 책임이 누구에게 있는지 설명할 수 있습니다.

<sub>관련 성취기준 [드코1 04-01-01] / [드코1 04-01-02]</sub>

> **생각 열기**  
> 친구에게 '이 코드 맞아?'라고 물으면 모를 때는 모른다고 합니다. 그런데 AI는 모르는 것도 아는 것처럼 또박또박 알려 주기도 합니다. 이 차이는 왜 생길까요?

<sub>교재에서 읽기: ■ 1. 생성형 AI는 어떻게 답을 만드는가 · ■ 2. 생성형 AI의 한계 · ■ 3. AI 활용의 기본 원칙 · ■ 4. 입력해서는 안 되는 정보 · ■ 5. 출처와 저작권</sub>

## ■ 6. 책임은 누구에게 있는가

AI가 알려 준 코드로 드론을 날렸는데 사고가 났다면, 그 책임은 누구에게 있을까요? AI는 책임을 질 수 없습니다. 코드를 검토하고 실행하기로 결정한 사람에게 책임이 있습니다.

### ====== 해 보기 ======

**사례를 두고 토의하기**

다음 상황을 읽고 모둠에서 토의해 봅시다.  
상황: 민수는 AI에게 '배터리가 부족하면 자동으로 착륙하는 코드'를 요청해 받은 코드를 그대로 드론에 넣었습니다. 그런데 AI가 만든 코드는 배터리를 %가 아니라 전압으로 읽고 있었고, 드론은 착륙하지 않고 떨어졌습니다.

1. 민수가 하지 않은 일은 무엇인가요?
2. 이 사고를 막으려면 어느 단계에서 무엇을 했어야 하나요?
3. 'AI가 잘못 알려 주었다'는 설명이 받아들여질 수 있을까요? 왜 그렇게 생각하나요?

### 핵심 정리

- 생성형 AI는 사실을 찾는 것이 아니라 그럴듯한 말을 이어 붙여 답을 만듭니다.
- 오류(환각), 최신성, 편향, 맥락 부족이라는 네 가지 한계가 있습니다.
- 확인·이해·밝힘·지킴·책임의 다섯 가지 원칙을 지켜 활용합니다.
- 개인정보는 입력하지 않으며, AI가 만든 결과의 책임은 사용자에게 있습니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 4-2 프롬프트 작성과 코드 생성

**학습 목표**

- 문제 해결 절차를 의사코드로 나타낼 수 있습니다.
- 좋은 프롬프트가 갖추어야 할 요소를 알고 작성할 수 있습니다.
- 생성된 결과에 대해 수정·보완을 요청할 수 있습니다.

<sub>관련 성취기준 [드코1 04-02-01] / [드코1 04-02-02]</sub>

> **생각 열기**  
> '드론 코드 짜 줘'라고 하면 엉뚱한 답이 옵니다. 사람에게 일을 부탁할 때처럼 AI에게도 무엇을 어떻게 해 달라는지 분명히 말해야 합니다. 그러려면 먼저 내가 무엇을 원하는지 스스로 정리해야 합니다.

## ■ 1. 의사코드로 생각 정리하기

의사코드(pseudocode)는 프로그램의 처리 절차를 사람의 말로 적은 것입니다. 문법을 지킬 필요가 없으므로 무엇을 할지에만 집중할 수 있습니다. 의사코드를 먼저 쓰면 AI에게 설명하기도, 결과를 검증하기도 쉬워집니다.

| 단계 | 할 일 |
|---|---|
| 1 | 무엇을 입력받는지 적습니다 |
| 2 | 어떤 순서로 처리할지 적습니다 |
| 3 | 조건이 갈리는 곳을 '만약 \~라면'으로 적습니다 |
| 4 | 무엇을 결과로 내놓을지 적습니다 |

<sub>표 4-3. 의사코드 쓰는 순서</sub>

> **짚고 가기 · 의사코드 예시: 안전 배송 비행**

```python
입력: 구간별 거리 목록, 최소 배터리 기준
드론에 연결한다
만약 배터리가 기준보다 적으면
    "비행 취소"를 출력하고 끝낸다
이륙한다
각 구간에 대하여 반복한다
    만약 배터리가 기준보다 적으면
        "복귀"를 출력하고 반복을 멈춘다
    그 구간만큼 전진한다
착륙한다
비행 기록을 저장한다
출력: 완료한 구간 수, 남은 배터리
```

\*이 의사코드는 1-5의 예제 1-20과 같은 내용입니다. 코드를 먼저 쓰지 않고 절차를 먼저 정리하면, 빠뜨린 조건을 쉽게 찾을 수 있습니다.\*

<sub>교재에서 읽기: ■ 2. 좋은 프롬프트의 다섯 요소 · ■ 3. 수정·보완 요청하기</sub>

## ■ 4. 의사코드를 코드로 옮기기

AI의 도움을 받든 직접 쓰든, 의사코드를 코드로 옮기는 과정은 같습니다. 아래 코드는 의사코드의 각 줄을 주석으로 옆에 붙여 두었습니다.

**예제 4-1. 의사코드를 코드로 옮기기**

In [ ]:
# 예제 4-1. 의사코드를 코드로 옮기기
from drone_sim import *

# 입력: 구간별 거리 목록, 최소 배터리 기준
plan = [120, 90, 150, 110]
LIMIT = 36

drone = Drone()
drone.sim_set_battery(38)                   # 가상 전용: 연습을 위해 배터리를 낮춰 둔다
drone.pair()                                # 드론에 연결한다

# 만약 배터리가 기준보다 적으면

    print("비행 취소: 배터리 부족")          #   "비행 취소"를 출력하고 끝낸다
else:
    # 이륙한다
    
    done = 0
    # 각 구간에 대하여 반복한다
    
        # 만약 배터리가 기준보다 적으면
        
            print(f"복귀: {n}번 구간부터 포기")  #     "복귀"를 출력하고 반복을 멈춘다
            # (여기에 반복을 멈추는 명령)
            
        # 그 구간만큼 전진한다
        
        done = done + 1
    # 착륙한다
    
    drone.sim_save_log()                    # 비행 기록을 저장한다
    print(f"완료한 구간: {done}개, 남은 배터리: {drone.get_battery()} %")   # 출력
drone.close()

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 38 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 120cm
[가상] 앞으로 90cm
[가상] 앞으로 150cm
복귀: 4번 구간부터 포기
[가상] 착륙
[가상] 비행 기록 8줄을 flight_log.csv 에 저장했습니다.
완료한 구간: 3개, 남은 배터리: 34 %
[가상] 연결을 닫았습니다.
```

의사코드의 각 줄이 코드의 어느 부분이 되었는지 하나씩 짚어 봅시다. 의사코드의 '끝낸다'는 코드에서 else가 되었습니다. 비행을 취소하면 이륙부터 착륙까지를 모두 건너뛰어야 하기 때문입니다. 의사코드와 코드가 한 줄씩 대응되면 잘 옮긴 것입니다.

### ====== 해 보기 ======

**의사코드 쓰고 바꾸기**

1. '배터리가 50% 아래로 떨어질 때까지 사각형을 되풀이해 그리는 비행'을 의사코드로 써 봅시다.
2. 표 4-4의 다섯 요소를 갖춘 프롬프트를 종이에 작성합니다.
3. AI를 쓸 수 있다면 실제로 요청해 보고, 받은 코드를 의사코드와 한 줄씩 비교합니다.
4. AI를 쓰지 않는다면 직접 코드로 옮긴 뒤, 짝과 바꿔 서로의 의사코드와 코드가 맞는지 확인합니다.

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 의사코드는 문법에 얽매이지 않고 처리 절차를 사람의 말로 적은 것입니다.
- 좋은 프롬프트에는 역할, 맥락, 할 일, 조건, 형식이 담깁니다.
- 쓸 수 있는 명령을 분명히 알려 주면 없는 명령을 지어내는 일이 줄어듭니다.
- 한 번에 원하는 답을 얻지 못하면 무엇이 문제인지 구체적으로 짚어 다시 요청합니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 4-3 AI 생성 결과 검증 및 개선

**학습 목표**

- AI가 생성한 코드를 단계적으로 검증할 수 있습니다.
- 오류를 찾아 고치고 그 근거를 설명할 수 있습니다.
- 검증 과정을 기록으로 남길 수 있습니다.

<sub>관련 성취기준 [드코1 04-03-01] / [드코1 04-03-02]</sub>

> **생각 열기**  
> AI가 준 코드가 깔끔해 보입니다. 주석도 잘 달려 있습니다. 그런데 그대로 드론에 넣어도 될까요? 보기 좋은 코드와 올바른 코드는 다릅니다.

<sub>교재에서 읽기: ■ 1. 검증의 네 단계</sub>

## ■ 2. 사례: AI가 만든 코드 검증하기

다음은 '목표 거리만큼 전진한 뒤 착륙하는 코드'를 요청해서 받았다고 가정한 코드입니다. 겨보기에는 문제가 없어 보입니다.

**예제 4-2. AI가 만들었다고 가정한 코드**

In [ ]:
# 예제 4-2. AI가 만들었다고 가정한 코드
# (빈칸 없음) 그대로 실행하고 결과를 살펴봅시다.
from drone_sim import *

target_m = 5                      # 5 m 를 가고 싶다

drone = Drone()
drone.pair()
drone.takeoff()
drone.move_forward(target_m)      # 목표 거리만큼 전진
print("도착 위치:", drone.get_pos_x(), "cm")
drone.land()
drone.close()
print("목표였던 거리: 5 m = 500 cm")

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 5cm
도착 위치: 5.0 cm
[가상] 착륙
[가상] 연결을 닫았습니다.
목표였던 거리: 5 m = 500 cm
```

4단계 검증에서 문제가 드러났습니다. 5 m를 가려 했는데 5 cm밖에 가지 못했습니다. move\_forward()의 단위는 적지 않으면 cm인데(표 1-10), m 단위 값을 그대로 넘겼기 때문입니다.

> **짚고 가기 · 왜 오류 문구가 없었을까요**  
> move\_forward(5)는 문법적으로 완전히 올바른 코드입니다. 5 cm를 가라는 명령이 제대로 실행되었을 뿐입니다. 컴퓨터는 '단위가 틀렸다'는 것을 알지 못합니다. 이처럼 오류 문구 없이 조용히 잘못 동작하는 경우가 가장 찾기 어렵고 위험합니다. 1-5에서 본 "mm" 단위 오류는 그나마 한 줄을 출력했지만, 이번에는 그런 한 줄조차 없습니다.

## ■ 3. 오류 고치기

코드론에듀의 move\_forward()는 단위를 두 번째 인자로 받습니다. 단위를 "m"로 알려 주면 고쳐집니다. 여기에 더해 변수 이름에 단위(\_m)를 드러내면 같은 실수를 막을 수 있습니다.

**예제 4-3. 단위를 바로잡고 안전장치를 더한 코드**

In [ ]:
# 예제 4-3. 단위를 바로잡고 안전장치를 더한 코드
from drone_sim import *

target_m = 5                               # 단위를 이름에 드러낸다
MIN_BATTERY = 30

drone = Drone()
drone.pair()
# [수정 2] 배터리 확인을 더한다

    print("배터리가 부족하여 비행을 취소합니다.")
else:
    drone.takeoff()
    # [수정 1] 단위를 m 로 알려 준다
    
    x = drone.get_pos_x()
    print("도착 위치:", x, "cm")
    print("목표 500 cm 에 30 cm 안쪽으로 도달했는가?", abs(x - 500) <= 30)
    drone.land()
drone.close()

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 5m
도착 위치: 500.0 cm
목표 500 cm 에 30 cm 안쪽으로 도달했는가? True
[가상] 착륙
[가상] 연결을 닫았습니다.
```

abs()는 부호를 떼고 크기만 남기는 함수입니다. 왜 x == 500으로 비교하지 않았을까요? 가상 드론은 정확히 500.0에 도착하지만, 실기체는 1-5에서 본 것처럼 센서 오차가 있어 딱 500이 나오지 않습니다. 그래서 허용할 범위를 정해 비교합니다. 또 5 m는 교실에서 날리기에 긴 거리입니다. 실기체로 옮길 때는 실습 공간에 맞게 거리를 줄입니다.

## ■ 4. 오류 문구 읽기

실행조차 되지 않는 코드를 받을 때도 있습니다. 이때는 오류 문구를 끝까지 읽는 것이 가장 빠른 해결 방법입니다.

**예제 4-4. 오류 문구 읽기**

In [ ]:
# 예제 4-4. 오류 문구 읽기
# (빈칸 없음) 그대로 실행하고 결과를 살펴봅시다.
route = [100, 150, 200]

for i in range(len(route)):
    leg = route[i] + route[i + 1]        # AI 가 만든 잘못된 줄
    print("두 구간의 합:", leg)

실행 결과

```
두 구간의 합: 250
두 구간의 합: 350
Traceback (most recent call last):
  ...
    leg = route[i] + route[i + 1]        # AI 가 만든 잘못된 줄
                     ~~~~~^^^^^^^
IndexError: list index out of range
```

코랩에서는 오류 문구의 윗부분 모양이 조금 다르게 보이지만 마지막 줄은 같습니다. 오류 문구는 아래에서부터 읽습니다. 마지막 줄은 오류의 종류(IndexError)와 내용(번호가 범위를 벗어남)을, 그 위의 \^ 표시는 문제가 난 자리인 route[i + 1]을 가리킵니다. 두 줄은 출력된 뒤 멈춘 것을 보면, 세 번째 반복에서 문제가 생겼다는 것도 알 수 있습니다.

route의 길이는 3이므로 번호는 0, 1, 2까지만 쓸 수 있는데, i가 2일 때 route[3]을 읽으려 한 것입니다. 3-3의 예제 3-9처럼 range(len(route) - 1)로 고치면 해결됩니다. 'AI 드론 지능형 제어'에서는 이런 오류가 나도 프로그램이 멈추지 않게 하는 예외 처리(try)를 배웁니다.

## ■ 5. 검증 기록 남기기

AI를 활용했다면 무엇을 어떻게 확인했는지 기록해야 합니다. 이 기록은 과제 평가의 근거가 되고, 나중에 같은 문제를 다시 만났을 때 도움이 됩니다.

| 항목 | 기록 내용 예시 |
|---|---|
| 요청한 내용 | 목표 거리만큼 전진한 뒤 착륙하는 코드를 요청함 |
| 받은 결과 | move\_forward(target\_m)을 쓴 10줄짜리 코드 |
| 1\~3단계 확인 | 읽고 이해함 / 명령은 모두 표 1-4에 있음 / 오류 없이 실행됨 |
| 4단계 확인 | 5를 넣었더니 도착 위치가 5.0 cm, 목표인 500 cm와 다름 |
| 수정 내용 | 단위 인자 "m"를 더하고 배터리 확인을 추가함 |
| 판단 근거 | 표 1-10에 move\_forward의 기본 단위가 cm로 되어 있음 |

<sub>표 4-7. 검증 기록표의 예</sub>

### ====== 해 보기 ======

**검증 기록표 작성하기**

1. 예제 4-2의 코드를 처음 받았다고 생각하고, 표 4-6의 네 단계를 차례로 직접 해 봅니다.
2. 단계마다 확인한 내용을 표 4-7 형식으로 기록합니다.
3. 짝과 기록표를 바꿔 읽고, 빠진 단계가 없는지 확인해 줍니다.
4. 'AI가 준 코드를 그대로 썼을 때 생길 수 있는 일'을 한 문장으로 적어 봅시다.

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- AI 코드는 읽기 → 존재 확인 → 실행 → 결과 확인의 네 단계로 검증합니다.
- 오류 문구 없이 조용히 잘못 동작하는 경우가 가장 찾기 어렵고 위험합니다.
- 변수 이름에 단위를 드러내고 단위 인자를 분명히 적으면 같은 실수를 막을 수 있습니다.
- 무엇을 어떻게 확인했는지 기록으로 남깁니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 4-4 기계학습 기초와 결정트리 활용

**학습 목표**

- 규칙 기반 판단과 기계학습의 차이를 설명할 수 있습니다.
- 결정트리를 학습시키고 판단 과정을 해석할 수 있습니다.
- 모델 평가 결과의 의미와 한계를 설명할 수 있습니다.

<sub>관련 성취기준 [드코1 04-04-01] / [드코1 04-04-02]</sub>

> **생각 열기**  
> Ⅱ단원에서 우리는 '풍속 5 이하'라는 기준을 직접 정했습니다. 그런데 그 5라는 숫자는 어디서 왔을까요? 또 3-1에서는 같은 바람이라도 속도에 따라 밀리는 거리가 달랐습니다. 지난 비행 기록이 많이 있다면, 컴퓨터가 스스로 알맞은 기준을 찾아낼 수는 없을까요?

<sub>교재에서 읽기: ■ 1. 두 가지 판단 방식</sub>

## ■ 2. 같은 문제를 두 방법으로

이번 문제는 '2 m 앞의 배송 지점에서 30 cm 안쪽에 도착할 수 있을까?'입니다. 먼저 사람이 기준을 정해 판단해 봅시다. 기준은 '풍속 3 m/s 이하면 가능'으로 정했습니다.

**예제 4-5. 규칙 기반 판단**

In [ ]:
# 예제 4-5. 규칙 기반 판단
LIMIT = 3.0                                   # 사람이 정한 기준: 풍속 3 m/s 이하

tests = [(2.1, 0.5), (3.6, 0.8), (3.6, 0.3), (5.0, 0.8)]   # (풍속, 속도)
for wind, speed in tests:
    # 풍속이 기준 이하이면 "가능", 아니면 "불가"
    result =
    print(f"풍속 {wind}, 속도 {speed} → {result}")

실행 결과

```
풍속 2.1, 속도 0.5 → 가능
풍속 3.6, 속도 0.8 → 불가
풍속 3.6, 속도 0.3 → 불가
풍속 5.0, 속도 0.8 → 불가
```

판단의 근거가 코드에 그대로 드러나 있어 왜 '불가'인지 설명하기 쉽습니다. 하지만 이 규칙은 속도를 전혀 보지 않습니다. 3.0이라는 숫자가 정말 알맞은지도 알 수 없습니다.

<sub>교재에서 읽기: ■ 3. 기계학습으로 기준 찾기</sub>

## ■ 4. 학습용 데이터 준비하기

기계학습에는 문제(특성)와 정답(레이블)이 짝지어진 데이터가 필요합니다. 여기서는 풍속과 속도가 문제이고, 실제로 배송 지점 30 cm 안에 도착했는지가 정답입니다. 정답은 사람이 정한 규칙이 아니라, 가상 드론을 60번 날려 본 결과에서 얻습니다.

**예제 4-6. 가상 비행으로 기록 모으기**

In [ ]:
# 예제 4-6. 가상 비행으로 기록 모으기
from drone_sim import *
import pandas as pd
import random

rng = random.Random(42)            # 같은 결과가 나오도록 난수를 고정
rows = []
for n in range(60):                # 가상 비행 60번
    wind = round(rng.uniform(0, 6), 1)
    speed = rng.choice([0.3, 0.5, 0.8])
    drone = Drone(verbose=False)
    drone.sim_set_wind(wind)
    drone.pair()
    drone.takeoff()
    drone.move_forward(200, speed=speed)          # 2 m 앞의 배송 지점으로
    x, y = drone.get_pos_x(), drone.get_pos_y()
    drone.land()
    drone.close()
    miss =        # 배송 지점에서 벗어난 거리(cm)
    # rows에 풍속·속도·성공(벗어난 거리가 30 이하이면 1) 기록 추가
    

df = pd.DataFrame(rows)
df.to_csv("flight_history.csv", index=False, encoding="utf-8-sig")
print(df.head())
print("\n전체", len(df), "번 중 성공:", df["성공"].sum(), "번")

실행 결과

```
    풍속   속도  성공
0  3.8  0.3   0
1  4.4  0.3   0
2  1.3  0.8   1
3  0.6  0.8   1
4  5.4  0.3   0

전체 60 번 중 성공: 33 번
```

rng.uniform(0, 6)은 0과 6 사이의 아무 수나, rng.choice(…)는 리스트에서 아무 값이나 하나를 골라 줍니다. int(miss &lt;= 30)은 True를 1로, False를 0으로 바꿉니다.

> **짚고 가기 · 왜 난수를 고정하나요**  
> random.Random(42)의 42는 난수의 출발점입니다. 이 값을 고정하면 누가 몇 번을 실행해도 똑같은 데이터가 만들어집니다. 실험 결과를 다른 사람이 확인할 수 있어야 하므로, 기계학습에서는 이렇게 고정하는 일이 많습니다.

## ■ 5. 학습용과 평가용으로 나누기

학습에 쓴 데이터로 성능을 평가하면 안 됩니다. 시험 문제를 미리 보고 시험을 치는 것과 같기 때문입니다. 그래서 데이터를 두 묶음으로 나눕니다.

**예제 4-7. 결정트리 학습시키기**

In [ ]:
# 예제 4-7. 결정트리 학습시키기
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv("flight_history.csv")
# 문제(특성): 풍속과 속도 두 열
X =
# 정답(레이블): 성공 열
y =

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)
print("학습용:", len(X_train), "번 / 평가용:", len(X_test), "번")

model = DecisionTreeClassifier(max_depth=3, random_state=42)
# 학습용 데이터로 모델 학습시키기: fit

# 평가용 데이터로 예측하기: predict
pred =
print("모델의 정확도:", round(accuracy_score(y_test, pred) * 100, 1), "%")

rule = (X_test["풍속"] <= 3.0).astype(int)    # 사람이 정한 규칙으로 판단
print("규칙의 정확도:", round(accuracy_score(y_test, rule) * 100, 1), "%")

실행 결과

```
학습용: 42 번 / 평가용: 18 번
모델의 정확도: 94.4 %
규칙의 정확도: 88.9 %
```

fit()은 '학습하라', predict()는 '예측하라'는 뜻입니다. 평가용 18번 가운데 모델은 17번, 규칙은 16번을 맞혔습니다. max\_depth=3은 '질문을 세 번까지만 이어 가라'는 제한입니다.

## ■ 6. 학습한 기준 들여다보기

결정트리의 가장 큰 장점은 무엇을 배웠는지 눈으로 볼 수 있다는 것입니다.

**예제 4-8. 결정트리 그려 보기**

In [ ]:
# 예제 4-8. 결정트리 그려 보기
import pandas as pd
import matplotlib.pyplot as plt
import font_setup
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

df = pd.read_csv("flight_history.csv")
X, y = df[["풍속", "속도"]], df["성공"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)
model = DecisionTreeClassifier(max_depth=3, random_state=42)
# 모델 학습시키기


plt.figure(figsize=(7, 4))
plot_tree(model, feature_names=["풍속", "속도"],
          class_names=["실패", "성공"], filled=True, fontsize=8)
plt.tight_layout()
plt.savefig("tree.png", dpi=150)
print("트리를 tree.png 로 저장했습니다.")

print("\n[각 조건이 판단에 쓰인 정도]")
for name, value in zip(["풍속", "속도"], model.feature_importances_):
    print(f"  {name}: {round(value * 100, 1)} %")

실행 결과

```
트리를 tree.png 로 저장했습니다.

[각 조건이 판단에 쓰인 정도]
  풍속: 89.6 %
  속도: 10.4 %
```

<sub>(그림 4-1. 데이터로부터 학습한 결정트리 — 교재 참고)</sub>

맨 위 상자를 보면 '풍속 ≤ 3.0'으로 먼저 나눕니다. 우리가 정한 규칙과 같은 값을 데이터에서 찾아낸 것입니다. 달라지는 것은 오른쪽 가지입니다. 풍속이 3을 넘으면 규칙은 무조건 '불가'인데, 트리는 한 번 더 묻습니다. 속도가 0.65보다 빠르고 풍속이 4.7 이하이면 '성공'이라고 판단합니다. 3-1에서 본 것처럼 빠르게 날면 바람을 맞는 시간이 짧기 때문입니다.

상자 안의 samples는 그 상자에 들어온 비행 수, value는 [실패 수, 성공 수], class는 그 상자의 판단입니다. feature\_importances\_는 어느 조건이 판단에 더 크게 쓰였는지 알려 주는데, 여기서는 풍속이 89.6%로 훨씬 큽니다.

## ■ 7. 두 방식 비교하기

**예제 4-9. 규칙과 모델의 판단 비교하기**

In [ ]:
# 예제 4-9. 규칙과 모델의 판단 비교하기
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("flight_history.csv")
X, y = df[["풍속", "속도"]], df["성공"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)
model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

test = pd.DataFrame({"풍속": [2.1, 3.6, 3.6, 5.0, 4.5],
                     "속도": [0.5, 0.8, 0.3, 0.8, 0.8]})
# 풍속이 3.0 이하이면 1, 아니면 0
test["규칙"] =
# 모델에게 예측시키기
test["모델"] =
# 규칙과 모델의 판단이 같은가?
test["일치"] =
print(test)

실행 결과

```
    풍속   속도  규칙  모델     일치
0  2.1  0.5   1   1   True
1  3.6  0.8   0   1  False
2  3.6  0.3   0   0   True
3  5.0  0.8   0   0   True
4  4.5  0.8   0   1  False
```

두 방식이 갈린 곳은 모두 '바람은 3을 넘게 불지만 0.8 m/s로 빠르게 나는' 경우입니다. 규칙은 속도를 보지 않으므로 '불가', 모델은 데이터에서 얻은 경향대로 '성공'이라고 했습니다. 어느 쪽을 믿을지는 다음 절의 한계를 알고 나서 정해야 합니다.

## ■ 8. 결과 해석의 한계

> **안전 수칙 · 정확도가 높다고 안전한 것은 아닙니다.**  
> - 이 모델은 가상 드론의 규칙을 배웠습니다. 데이터가 가상 드론에서 나왔기 때문입니다. 실기체의 실제 비행 기록으로 다시 학습시키기 전에는 실기체 판단에 쓸 수 없습니다.
> - 데이터에 없던 상황은 판단할 수 없습니다. 풍속 6 m/s를 넘는 비행이나 비 오는 날은 학습한 적이 없습니다.
> - 평가용은 18번뿐입니다. 한 번만 틀려도 정확도가 5.6%씩 바뀝니다. 모델과 규칙의 차이도 정답 하나 차이입니다.
> - 모델은 왜 그렇게 판단했는지 책임지지 않습니다. 비행 여부의 최종 결정과 그 책임은 사람에게 있습니다.

그래서 실제 드론 운용에서는 기계학습의 판단을 참고 자료로 쓰고, 최종 결정은 사람이 공식 기상 정보와 현장 상황을 확인해 내립니다.

### ====== 해 보기 ======

**데이터를 바꾸면 기준도 바뀔까요**

1. 예제 4-6에서 range(60)을 range(15)로 줄여 다시 실행해 봅니다.
2. 예제 4-7과 4-8을 다시 실행해 정확도와 트리 모양이 어떻게 달라지는지 봅니다.
3. 데이터가 적을 때 어떤 문제가 생기는지 모둠에서 이야기합니다.
4. max\_depth를 1과 5로 바꿔 보고, 트리가 어떻게 달라지는지 확인합니다.
5. '데이터가 많으면 언제나 좋은가?'라는 질문을 두고 토의해 봅시다.

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 규칙 기반은 사람이 기준을 정하고, 기계학습은 데이터에서 기준을 찾습니다.
- 기계학습에는 특성과 레이블이 짝지어진 데이터가 필요하며, 학습용과 평가용을 나눕니다.
- 결정트리는 판단 과정을 그림으로 확인할 수 있어 해석하기 쉽습니다.
- 모델은 데이터가 나온 환경만큼만 믿을 수 있으며, 최종 결정과 책임은 사람에게 있습니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# Ⅳ단원 마무리

### 배운 내용 한눈에 보기

| 소단원 | 꼭 기억할 것 |
|---|---|
| 4-1 AI 활용 원칙 | 생성형 AI는 그럴듯한 말을 이어 붙여 답을 만듭니다. 결과를 확인하고, 개인정보는 넣지 않으며, 책임은 사용자에게 있습니다. |
| 4-2 프롬프트 작성과 코드 생성 | 의사코드로 절차를 정리한 뒤 역할·맥락·할 일·조건·형식을 갖춘 프롬프트를 씁니다. 쓸 수 있는 명령을 분명히 알려 줍니다. |
| 4-3 AI 생성 결과 검증 및 개선 | 읽기 → 존재 확인 → 실행 → 결과 확인의 네 단계로 검증합니다. 오류 문구 없이 잘못 동작하는 코드를 특히 조심합니다. |
| 4-4 기계학습 기초와 결정트리 활용 | 결정트리는 데이터에서 기준을 찾고 그 과정을 그림으로 보여 줍니다. 모델은 데이터가 나온 환경만큼만 믿을 수 있으며, 최종 결정은 사람이 내립니다. |

### 스스로 점검하기

단원을 시작할 때 세운 목표를 얼마나 이루었는지 표시해 봅니다. ○는 '혼자 할 수 있다', △는 '책을 보면 할 수 있다', ✕는 '아직 어렵다'입니다.

| 점검 내용 | ○ △ ✕ |
|---|---|
| 생성형 AI의 특성과 한계를 이해하고 활용 원칙을 지킬 수 있다. |  |
| 문제를 의사코드로 구조화하고 효과적인 프롬프트를 작성할 수 있다. |  |
| AI가 생성한 코드를 검증하고 오류를 고쳐 작동 원리를 설명할 수 있다. |  |
| 규칙 기반 판단과 결정트리의 판단을 비교하고 한계를 설명할 수 있다. |  |

△나 ✕에 표시한 항목은 관련 소단원의 핵심 정리와 예제로 돌아가 다시 살펴봅니다.